# 📈 Stock / Commodity Forecast

Questo notebook scarica una serie temporale da Yahoo Finance e produce previsioni con un **pipeline di model selection**:

1. **Train/Validation split** – le ultime *H* osservazioni sono usate come validation set
2. **Training & Evaluation** – ogni modello viene trainato sul train set e valutato sul validation
3. **Selezione** – il modello migliore (MAPE più basso) viene selezionato
4. **Retrain & Forecast** – il vincitore viene ritrainato su tutta la serie e produce la previsione finale

### Modelli disponibili
| Modello | Tipo |
|---|---|
| **Chronos** | Amazon, zero-shot transformer |
| **N-HiTS** | NeuralForecast, trained on-the-fly |
| **N-BEATS** | NeuralForecast, trained on-the-fly |

---
### Ticker di esempio
| Asset | Ticker |
|---|---|
| Apple | `AAPL` |
| S&P 500 | `^GSPC` |
| Oro | `GC=F` |
| Petrolio WTI | `CL=F` |
| Bitcoin | `BTC-USD` |
| EUR/USD | `EURUSD=X` |

In [1]:
import sys

sys.path.insert(0, "../src")

from stock_forecast.data import download_series, get_ticker_info, PRESET_TICKERS
from stock_forecast.models import run_pipeline, run_all, run_chronos, run_nhits, run_nbeats
from stock_forecast.plot import plot_forecast, print_summary, print_validation_table

## 1 · Configurazione

In [ ]:
# ── Parametri principali ──────────────────────────────────────────────
TICKER = "CL=F"  # Yahoo Finance ticker   (es: AAPL, ^GSPC, GC=F, BTC-USD)
PERIOD = "max"  # storico scaricato       (6mo | 1y | 2y | 5y | 10y | max)
INTERVAL = "1d"  # granularità             (1d | 1wk | 1mo)
HORIZON = 15  # giorni / barre da prevedere

# ── Opzioni modelli ───────────────────────────────────────────────────
CHRONOS_SIZE = "large"  # tiny | mini | small | base | large
MAX_STEPS = 500  # iterazioni di training per N-HiTS e N-BEATS
MODELS = ["N-HiTS", "N-BEATS", "Chronos", "Auto-ARIMA", "XGBoost"]  # modelli da confrontare
SELECTION_METRIC = "mae"  # metrica per la selezione (mae | rmse | mape)

print(f"Ticker  : {TICKER}  |  Periodo: {PERIOD}  |  Orizzonte: {HORIZON} barre")
print(f"Modelli : {', '.join(MODELS)}")
print(f"Metrica : {SELECTION_METRIC}")

Ticker  : CL=F  |  Periodo: max  |  Orizzonte: 15 barre
Modelli : N-HiTS, N-BEATS, Chronos, Auto-ARIMA, XGBoost
Metrica : mae


## 2 · Download dati

In [3]:
info = get_ticker_info(TICKER)
print(f"\nAsset   : {info.get('name', TICKER)}")
print(f"Ticker  : {info['ticker']}")
print(f"Currency: {info.get('currency', '—')}")
print(f"Sector  : {info.get('sector', '—')}")

df = download_series(TICKER, period=PERIOD, interval=INTERVAL)
print(f"\nSerie scaricata: {len(df)} osservazioni")
print(f"Da {df['ds'].min().date()}  →  {df['ds'].max().date()}")
df.tail()


Asset   : Crude Oil Jul 26
Ticker  : CL=F
Currency: USD
Sector  : —

Serie scaricata: 6472 osservazioni
Da 2000-08-23  →  2026-06-03


Price,ds,Close,High,Low,Open,Volume,y
6467,2026-05-28,88.900002,92.519997,87.110001,89.110001,235145,88.900002
6468,2026-05-29,87.360001,89.019997,86.349998,88.550003,267803,87.360001
6469,2026-06-01,92.160004,94.779999,88.449997,88.500000,322039,92.160004
6470,2026-06-02,93.760002,94.000000,90.120003,92.449997,322039,93.760002
6471,2026-06-03,96.400002,97.000000,93.449997,93.449997,83235,96.400002


## 3 · Model Selection Pipeline

Il pipeline:
1. Split train / validation (ultime `HORIZON` osservazioni)
2. Train & evaluate ogni modello sul validation set
3. Seleziona il migliore per MAPE
4. Retrain sul dataset completo → previsione finale

In [4]:
val_results, best_forecast = run_pipeline(
    df,
    horizon=HORIZON,
    models=MODELS,
    chronos_size=CHRONOS_SIZE,
    max_steps=MAX_STEPS,
    selection_metric=SELECTION_METRIC,
)


from stock_forecast.models import select_best_model

best_name = select_best_model(val_results, metric=SELECTION_METRIC)
print_validation_table(val_results, best_name=best_name)

print_summary({best_forecast.model_name: best_forecast}, ticker_info=info)

Seed set to 1


╔═══════════════════════════════════════════════════════╗
║          MODEL SELECTION PIPELINE                    ║
╚═══════════════════════════════════════════════════════╝

  Series length : 6472 observations
  Horizon       : 15 steps
  Val set size  : 15 (last 15 obs)
  Metric        : mae

──────────── Models to validate: N-HiTS, N-BEATS, Chronos, Auto-ARIMA, XGBoost ────────────

── Validating: N-HiTS ─────────────────────────────────
  🏋️  Training N-HiTS (max_steps=700) …


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MQLoss        │      3 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  2.6 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 2.6 M                                                                                            
Non-trainable params: 3                                                                                            
Total params: 2.6 M                                                                                                
Total estimated model params size (MB): 10                                                                         
Modules in train mode: 34                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.


  🔮 Forecasting 15 steps …


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Output()

  ✅ MAE=9.2734  RMSE=11.0305  MAPE=9.98%

── Validating: Chronos ─────────────────────────────────
  ⏳ Loading Chronos [large] on cpu …
  🔮 Forecasting 15 steps (ChronosPipeline) …
  ✅ MAE=6.1678  RMSE=7.3843  MAPE=6.53%

── Validating: Auto-ARIMA ─────────────────────────────────
  📊 ADF test p-value: 0.063578 → non-stationary
  📐 Differencing order d=1
  🔍 Searching ARIMA orders (p=0..5, d=1, q=0..5) …
  ✅ Best order: ARIMA(5, 1, 2) (AIC=26055.39)
  ⚠️  Auto-ARIMA failed: 'numpy.ndarray' object has no attribute 'iloc'

── Validating: XGBoost ─────────────────────────────────
  🌲 Training XGBoost (lags=45, estimators=500) …
  🔮 Forecasting 15 steps (recursive) …
  ✅ MAE=9.3819  RMSE=10.9992  MAPE=10.08%

═══════════════════════════════════════════════════════
  🏆 Best model: Chronos
  🔄 Retraining on full series (6472 obs) …
═══════════════════════════════════════════════════════

  ⏳ Loading Chronos [large] on cpu …
  🔮 Forecasting 15 steps (ChronosPipeline) …

══════════════════════

## 4 · Grafico interattivo

In [5]:
fig = plot_forecast(
    best_forecast,
    ticker=info.get("name", TICKER),
    last_n_days=180,
    show_volume=True,
)
fig.show()

## 5 · Salva grafico

In [6]:
import os

os.makedirs("../outputs", exist_ok=True)
safe_ticker = TICKER.replace("=", "_").replace("^", "")
out_path = f"../outputs/{safe_ticker}_forecast.html"
fig.write_html(out_path)
print(f"Grafico salvato in: {out_path}")

Grafico salvato in: ../outputs/CL_F_forecast.html
